In [1]:
import sys
sys.path.append('../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import sqlite3
from data_io import load_from_sqlite
from config import RECOMMENDER_WEIGHTS, MIN_EVENTS_FOR_RECOMMENDATION, LOW_SAMPLE_LAPS_THRESHOLD

DB_PATH = '../data/processed/wec_bop.db'

event_model = load_from_sqlite("SELECT * FROM event_model_features", DB_PATH)

print(f"Loaded: {len(event_model):,} rows")
print(f"Classes: {event_model['class_normalized'].value_counts().to_dict()}")

Loaded: 305 rows
Classes: {'HYPERCAR': 179, 'GT3': 126}


In [2]:
SEASON_GROUP = [
    'year', 'class_normalized', 'homologation_group',
    'manufacturer', 'car_model_key'
]

season_agg = (
    event_model
    .groupby(SEASON_GROUP, as_index=False)
    .agg(
        events_count      = ('event', 'nunique'),
        clean_laps_total  = ('clean_laps_count', 'sum'),
        avg_delta         = ('baseline_delta', 'mean'),
        median_delta      = ('baseline_delta', 'median'),
        std_delta         = ('baseline_delta', 'std'),       # track stability
        avg_consistency   = ('consistency_score', 'mean'),
        avg_long_run      = ('long_run_score', 'median'),
        avg_track_balance = ('track_balance_score', 'mean'),
    )
    .round(4)
)

print(f"Season aggregations: {len(season_agg):,} rows")
print(season_agg.groupby(['class_normalized', 'year']).size())

Season aggregations: 49 rows
class_normalized  year
GT3               2024    9
                  2025    9
HYPERCAR          2021    3
                  2022    4
                  2023    7
                  2024    9
                  2025    8
dtype: int64


In [3]:
def normalize_within_group(df, col, group_cols, new_col, invert=False):
    """
    Нормализует колонку в диапазон -1..1 относительно группы.
    invert=True: меньшее значение → лучше (например delta)
    invert=False: большее значение → лучше (например consistency)
    """
    df = df.copy()
    grouped = df.groupby(group_cols)[col]
    col_min = grouped.transform('min')
    col_max = grouped.transform('max')
    denom = (col_max - col_min).replace(0, np.nan)

    normalized = (df[col] - col_min) / denom  # 0..1

    if invert:
        normalized = 1 - normalized

    # -1..1
    df[new_col] = (normalized * 2 - 1).clip(-1, 1)
    return df


GROUP_NORM = ['year', 'class_normalized', 'homologation_group']

# pace: low avg_delta → better → invert=True
season_agg = normalize_within_group(
    season_agg, 'avg_delta', GROUP_NORM, 'pace_norm', invert=True
)

# consistency: more → better → invert=False
season_agg = normalize_within_group(
    season_agg, 'avg_consistency', GROUP_NORM, 'consistency_norm', invert=False
)

# long_run: low slope → better (low degradation) → invert=True
season_agg = normalize_within_group(
    season_agg, 'avg_long_run', GROUP_NORM, 'long_run_norm', invert=True
)

# stability: low std_delta → stability → better → invert=True
season_agg = normalize_within_group(
    season_agg, 'std_delta', GROUP_NORM, 'stability_norm', invert=True
)

print("=== Normalized components sample ===")
print(season_agg[['car_model_key', 'year',
                   'pace_norm', 'consistency_norm',
                   'long_run_norm', 'stability_norm']].head(8).to_string(index=False))

=== Normalized components sample ===
  car_model_key  year  pace_norm  consistency_norm  long_run_norm  stability_norm
     Alpine_LMH  2021  -0.046435          0.490665       0.495987        0.296364
Glickenhaus_LMH  2021  -1.000000         -1.000000       1.000000       -1.000000
     Toyota_LMH  2021   1.000000          1.000000      -1.000000        1.000000
     Alpine_LMH  2022  -0.716238         -0.493718       1.000000        0.246010
Glickenhaus_LMH  2022  -1.000000         -1.000000      -0.961379       -0.079416
    Peugeot_LMH  2022  -0.528320          1.000000      -1.000000        1.000000
     Toyota_LMH  2022   1.000000          0.491857      -1.000000       -1.000000
  Cadillac_LMDh  2023   1.000000         -1.000000       1.000000       -1.000000


In [8]:
def score_to_points(df):
    """
    Переводим balance_score в recommendation_points.
    balance_score уже в диапазоне -1..1,
    умножаем на 5 для шкалы -5..+5.
    Положительный = overperforming = penalty.
    Отрицательный = underperforming = relief.
    """
    df = df.copy()
    df['recommendation_points'] = (df['balance_score'] * 5).round(1).clip(-5, 5)
    return df

season_agg = score_to_points(season_agg)

print("=== recommendation_points distribution ===")
print(season_agg['recommendation_points'].describe().round(2))
print("\n=== value counts (binned) ===")
print(pd.cut(season_agg['recommendation_points'],
             bins=[-5, -3, -1, 1, 3, 5],
             labels=['strong_relief', 'mild_relief',
                     'no_change', 'mild_penalty', 'strong_penalty']
             ).value_counts())

=== recommendation_points distribution ===
count    49.00
mean      0.21
std       2.52
min      -5.00
25%      -2.00
50%       0.90
75%       2.00
max       4.50
Name: recommendation_points, dtype: float64

=== value counts (binned) ===
recommendation_points
mild_penalty      17
no_change         10
mild_relief        8
strong_relief      7
strong_penalty     6
Name: count, dtype: int64


In [9]:
def assign_label(points):
    if points >= 3.0:
        return 'strong_penalty'
    elif points >= 1.0:
        return 'mild_penalty'
    elif points <= -3.0:
        return 'strong_relief'
    elif points <= -1.0:
        return 'mild_relief'
    else:
        return 'no_change'

season_agg['recommendation_label'] = season_agg['recommendation_points'].apply(assign_label)

print("=== Labels distribution ===")
print(season_agg['recommendation_label'].value_counts())

=== Labels distribution ===
recommendation_label
mild_penalty      17
no_change          9
strong_relief      8
mild_relief        8
strong_penalty     7
Name: count, dtype: int64


In [10]:
def calc_confidence(row):
    score = 1.0

    # Штраф за малое число событий
    if row['events_count'] < MIN_EVENTS_FOR_RECOMMENDATION:
        score *= 0.5
    elif row['events_count'] < 4:
        score *= 0.75

    # Штраф за малое число кругов
    if row['clean_laps_total'] < LOW_SAMPLE_LAPS_THRESHOLD:
        score *= 0.6
    elif row['clean_laps_total'] < LOW_SAMPLE_LAPS_THRESHOLD * 2:
        score *= 0.85

    # Штраф за высокую нестабильность между трассами
    if pd.notna(row['std_delta']) and row['std_delta'] > 1.0:
        score *= 0.8

    return round(min(score, 1.0), 2)

season_agg['confidence_score'] = season_agg.apply(calc_confidence, axis=1)

print("=== Confidence score distribution ===")
print(season_agg['confidence_score'].describe().round(2))
print("\n=== Low confidence models ===")
print(season_agg[season_agg['confidence_score'] < 0.7][
    ['car_model_key', 'year', 'events_count',
     'clean_laps_total', 'confidence_score']
].sort_values('confidence_score').to_string(index=False))

=== Confidence score distribution ===
count    49.00
mean      0.96
std       0.12
min       0.51
25%       1.00
50%       1.00
75%       1.00
max       1.00
Name: confidence_score, dtype: float64

=== Low confidence models ===
       car_model_key  year  events_count  clean_laps_total  confidence_score
     Glickenhaus_LMH  2021             2               439              0.51
     Glickenhaus_LMH  2023             3               333              0.51
Isotta Fraschini_LMH  2024             3               357              0.64


In [7]:
def build_explanation(row):
    label_map = {
        'strong_penalty': 'системно БЫСТРЕЕ baseline → рекомендована нагрузка',
        'mild_penalty':   'немного быстрее baseline → лёгкая нагрузка',
        'no_change':      'в балансе с baseline → корректировка не нужна',
        'mild_relief':    'немного медленнее baseline → лёгкое облегчение',
        'strong_relief':  'системно МЕДЛЕННЕЕ baseline → рекомендовано облегчение',
    }
    return (
        f"{label_map[row['recommendation_label']]}. "
        f"Avg delta: {row['avg_delta']:+.3f}s, "
        f"events: {int(row['events_count'])}, "
        f"laps: {int(row['clean_laps_total'])}, "
        f"consistency: {row['avg_consistency']:.2f}, "
        f"long_run: {row['avg_long_run']:+.4f}s/lap, "
        f"confidence: {row['confidence_score']:.2f}"
    )

season_agg['explanation_text'] = season_agg.apply(build_explanation, axis=1)

# Превью
print("=== Sample explanations ===")
for _, row in season_agg[season_agg['class_normalized'] == 'HYPERCAR'].head(4).iterrows():
    print(f"\n{row['car_model_key']} ({row['year']}):")
    print(f"  Points: {row['recommendation_points']:+.1f} | Label: {row['recommendation_label']}")
    print(f"  {row['explanation_text']}")

=== Sample explanations ===

Alpine_LMH (2021):
  Points: +1.7 | Label: mild_penalty
  немного быстрее baseline → лёгкая нагрузка. Avg delta: +0.366s, events: 5, laps: 954, consistency: 0.57, long_run: +0.0150s/lap, confidence: 1.00

Glickenhaus_LMH (2021):
  Points: -5.0 | Label: strong_relief
  системно МЕДЛЕННЕЕ baseline → рекомендовано облегчение. Avg delta: +0.918s, events: 2, laps: 439, consistency: 0.41, long_run: -0.0007s/lap, confidence: 0.51

Toyota_LMH (2021):
  Points: +5.0 | Label: strong_penalty
  системно БЫСТРЕЕ baseline → рекомендована нагрузка. Avg delta: -0.240s, events: 5, laps: 1872, consistency: 0.62, long_run: +0.0616s/lap, confidence: 1.00

Alpine_LMH (2022):
  Points: +0.6 | Label: no_change
  в балансе с baseline → корректировка не нужна. Avg delta: +0.433s, events: 6, laps: 1068, consistency: 0.47, long_run: -0.0632s/lap, confidence: 1.00
